In [1]:
from google.colab import drive
import os
from scipy import stats
import numpy as np
import pandas as pd

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
BASE = '/content/drive/MyDrive'
AU_DIR = f'{BASE}/au_activity'

REAL = f'{AU_DIR}/real'
PSEUDO = f'{AU_DIR}/pseudopairings'
WLCC = f'{AU_DIR}/wlcc'

AUS = ['AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU07', 'AU09', 'AU10', 'AU11',
       'AU12', 'AU14', 'AU15', 'AU17', 'AU20', 'AU23', 'AU24', 'AU25', 'AU26',
       'AU28', 'AU43']

In [4]:
# standardized diff between means of two groups
def cohens_d(a, b):
  a, b = a.dropna(), b.dropna()

  na, nb = len(a), len(b)
  a_var, b_var = a.var(ddof=1), b.var(ddof=1)

  pooled_sd = np.sqrt(((na-1)*a_var + (nb-1)*b_var)/(na+nb-2))
  diff = (a.mean() - b.mean()) / pooled_sd
  return diff

In [5]:
# validate the mean synchrony of all AUs between real pairs and ensure
# statistical significance upon comparison to means of pseudopairings

if os.path.exists(f'{AU_DIR}/val_results.csv'):
  print(f'val_results.csv found in drive')
  print('loading file...')
  val_res = pd.read_csv(f'{AU_DIR}/val_results.csv')
  print(f'file loaded!')
else:
  print(f'val_results.csv not found in drive')
  print(f'creating file...')

  metrics = ['grand_aver', 'lag_zero_mean', 'peak_sync_mean']

  records = []

  for au in AUS:
    print(f'now processing {au}...')

    df = pd.read_csv(f'{WLCC}/{au}.csv')
    real = df[df['type'] == 'real']
    pseudo = df[df['type'] == 'pseudo']

    for metric in metrics:
      a = real[metric].dropna()
      b = pseudo[metric].dropna()

      w = stats.ttest_ind(a, b, equal_var=False)

      records.append({
          'au': au,
          'metric': metric,
          'real_mean': a.mean(),
          'real_sd': a.std(),
          'pseudo_mean': b.mean(),
          'pseudo_sd': b.std(),
          'welch_t': w.statistic,
          'welch_df': getattr(w, 'df', np.nan),
          'welch_p': w.pvalue,
          'cohens_d': cohens_d(a, b)
      })

      print('statisical analysis complete')
      print(f'p value: {w.pvalue}')

  val_res = pd.DataFrame(records)
  val_res.to_csv(f'{AU_DIR}/val_results.csv', index=False)

  print(f'file created!')

val_res.head()

val_results.csv found in drive
loading file...
file loaded!


,au,metric,real_mean,real_sd,pseudo_mean,pseudo_sd,welch_t,welch_df,welch_p,cohens_d,welch_p_adj
0,AU01,grand_aver,0.172252,0.019820,0.167325,0.019194,1.875282,213.954730,0.062116,0.252160,0.177475
1,AU01,lag_zero_mean,0.173826,0.027102,0.167104,0.025060,1.916670,216.790078,0.056595,0.256545,0.087069
2,AU01,peak_sync_mean,0.397620,0.026704,0.390493,0.026304,1.994767,212.715458,0.047345,0.268680,0.157816
3,AU02,grand_aver,0.172976,0.023466,0.166568,0.019888,2.201740,219.741849,0.028724,0.292197,0.114896
4,AU02,lag_zero_mean,0.175861,0.027701,0.163555,0.024990,3.475828,217.953483,0.000614,0.464106,0.002458


In [6]:
# now, check for directional lead between conversations: who leads the convo,
# and does that track with who enjoyed it more? correlate directional metric
# with enjoy_diff across real dyads. checks both pearson (linear) and spearman
# (monotonic non-linear) relationships

if os.path.exists(f'{AU_DIR}/directional_results.csv'):
  print(f'directional_results.csv found in drive')
  print(f'loading file...')
  dir_res = pd.read_csv(f'{AU_DIR}/directional_results.csv')
  print(f'file loaded!')
else:
  print(f'directional_results.csv not found in drive')
  print(f'creating file...')

  metrics = ['signed_lead_mean', 'best_lag_mean']

  records = []

  for au in AUS:
    df = pd.read_csv(f'{WLCC}/{au}.csv')
    real = df[df['type'] == 'real']

    for metric in metrics:
      x = real['enjoy_diff']
      y = real[metric]

      mask = x.notna() & y.notna()
      x, y, = x[mask], y[mask]

      # correlation: sign = who leads, magnitude = association strength
      pr, pp = stats.pearsonr(x, y)
      sr, sp = stats.spearmanr(x, y)

      # regression: slope = how much (lead shift per unit of enjoy_diff)
      # r2 = variance in data explained by regression
      slope, intercept, r, rp, se = stats.linregress(x, y)

      # sign-agreement: fraction of dyads where who-led matches who-enjoyed-more
      sign_agree = (np.sign(x) == np.sign(y)).mean()

      records.append({
          'au': au,
          'metric': metric,
          'n': len(x),
          'pearson_r': pr,
          'pearson_p': pp,
          'spearman_r': sr,
          'spearman_p': sp,
          'slope': slope,
          'r2': r**2,
          'slope_p': rp,
          'sign_agree': sign_agree
      })

  dir_res = pd.DataFrame(records)
  dir_res.to_csv(f'{AU_DIR}/directional_results.csv', index=False)

  print(f'file created!')

dir_res.head()

directional_results.csv found in drive
loading file...
file loaded!


,au,metric,n,pearson_r,pearson_p,spearman_r,spearman_p,slope,r2,slope_p,sign_agree,spearman_p_adj
0,AU01,signed_lead_mean,122,-0.084494,0.354802,-0.081680,0.371110,-0.000631,0.007139,0.354802,0.500000,0.926766
1,AU01,best_lag_mean,122,-0.061335,0.502138,-0.055445,0.544136,-0.006324,0.003762,0.502138,0.508197,0.739037
2,AU02,signed_lead_mean,122,0.027409,0.764418,-0.019251,0.833306,0.000226,0.000751,0.764418,0.500000,0.926766
3,AU02,best_lag_mean,122,-0.019007,0.835389,-0.057902,0.526408,-0.002026,0.000361,0.835389,0.475410,0.739037
4,AU04,signed_lead_mean,122,-0.176050,0.052417,-0.188794,0.037288,-0.001210,0.030994,0.052417,0.393443,0.607228


In [7]:
# adjust p-values for running an inference test on 20 AUs simultaneously
# by using false-discovery-rate (FDR) p-value correction on all tests

for metric in val_res['metric'].unique():
  rows = val_res['metric'] == metric
  val_res.loc[rows, 'welch_p_adj'] = stats.false_discovery_control(val_res.loc[rows, 'welch_p'])

for metric in dir_res['metric'].unique():
  rows = dir_res['metric'] == metric
  dir_res.loc[rows, 'spearman_p_adj'] = stats.false_discovery_control(dir_res.loc[rows, 'spearman_p'])

val_res.to_csv(f'{AU_DIR}/val_results.csv', index=False)
dir_res.to_csv(f'{AU_DIR}/directional_results.csv', index=False)

In [8]:
# checking to ensure that pseudopairs have no actual directional synchrony
# if not, results should be doubted and reevaluated

if os.path.exists(f'{AU_DIR}/directional_null_results.csv'):
  print(f'directional_null_results.csv found in drive')
  print(f'loading df...')
  null_dir_res = pd.read_csv(f'{AU_DIR}/directional_null_results.csv')
  print(f'df loaded!')
else:
  print(f'directional_null_results.csv not found in drive')
  print(f'building df...')

  metrics = ['signed_lead_mean', 'best_lag_mean']
  records = []

  for au in AUS:
    df = pd.read_csv(f'{WLCC}/{au}.csv')
    pseudo = df[df['type'] == 'pseudo']

    for metric in metrics:
      v = pseudo[metric].dropna()
      # does avg ≠ 0?
      t, p = stats.ttest_1samp(v, 0.0)

      records.append({
          'type': 'pseudo',
          'au': au,
          'metric': metric,
          'n': len(v),
          'mean': v.mean(),
          't': t,
          'p': p,
      })

  null_dir_res = pd.DataFrame(records)
  null_dir_res.to_csv(f'{AU_DIR}/directional_null_results.csv', index=False)

  print(f'df built!')

null_dir_res.head()

directional_null_results.csv found in drive
loading df...
df loaded!


,type,au,metric,n,mean,t,p
0,pseudo,AU01,signed_lead_mean,100,-0.000855,-0.313159,0.754819
1,pseudo,AU01,best_lag_mean,100,0.036739,0.943889,0.347524
2,pseudo,AU02,signed_lead_mean,100,-0.003571,-1.414149,0.160455
3,pseudo,AU02,best_lag_mean,100,-0.045866,-1.102804,0.272786
4,pseudo,AU04,signed_lead_mean,100,0.000776,0.226939,0.820939


In [9]:
ALPHA = 0.05

# validation: is real synchrony greater than by chance across AUs?
# our main metric to check for is grand_aver; however, check for
# significant post-fdr p-values across other metrics

print('=' * 80)
print(f'{'\t' * 2} VALIDATION (grand average, real vs pseudo)')
print(f'{'\t' * 3}    using alpha of a={ALPHA}')
print('=' * 80)
print('\n')

v = val_res[val_res['metric'] == 'grand_aver']
sig = v[v['welch_p_adj'] < ALPHA]
dropped = v[(v['welch_p'] < ALPHA) & (v['welch_p_adj'] >= ALPHA)]

print(f'post-FDR-correction: {len(sig)}/{len(v)} AUs w/ significant synchrony\n')
if len(sig) > 0:
  for r in sig.itertuples():
    print(f'{r.au}:')
    print(f'\tmean: {r.real_mean:.4f} (pseudo mean: {r.pseudo_mean:.4f})')
    print(f'\tstandardized difference: {r.cohens_d:.4f}')
    print(f'\tp-values: pre-FDR {r.welch_p:.6f} & post-FDR {r.welch_p_adj:.6f}')

if len(dropped) > 0:
  print(f'\n{len(dropped)} AUs with significant pre-correction p-values.\n')
  for r in dropped.itertuples():
    print(f'{r.au}:')
    print(f'\tmean: {r.real_mean:.4f} (pseudo mean: {r.pseudo_mean:.4f})')
    print(f'\tstandardized difference: {r.cohens_d:.4f}')
    print(f'\tp-values: pre-FDR {r.welch_p:.6f} & post-FDR {r.welch_p_adj:.6f}')

sig2 = val_res[(val_res['metric'] == 'lag_zero_mean') & (val_res['welch_p_adj'] < ALPHA)]
sig3 = val_res[(val_res['metric'] == 'peak_sync_mean') & (val_res['welch_p_adj'] < ALPHA)]

print('\n')
print('=' * 80)
print(f'{'\t' * 2} VALIDATION (lag zero mean, real vs pseudo)')
print(f'{'\t' * 3}    using alpha of a={ALPHA}')
print('=' * 80)
print('\n')

print(f'{len(sig2)}/20 AUs with significant lag zero mean p-values, post-FDR correction\n')
if len(sig2) > 0:
  for r in sig2.itertuples():
    print(f'{r.au}:')
    print(f'\tmean: {r.real_mean:.4f} (pseudo mean: {r.pseudo_mean:.4f})')
    print(f'\tstandardized difference: {r.cohens_d:.4f}')
    print(f'\tp-values: pre-FDR {r.welch_p:.6f} & post-FDR {r.welch_p_adj:.6f}')

print('\n')
print('=' * 80)
print(f'{'\t' * 2} VALIDATION (peak synchrony mean, real vs pseudo)')
print(f'{'\t' * 3}      using alpha of a={ALPHA}')
print('=' * 80)


print(f'\n{len(sig3)}/20 AUs with significant peak synchrony mean p-values, post-FDR correction\n')
if len(sig3) > 0:
  for r in sig3.itertuples():
    print(f'{r.au}:')
    print(f'\tmean: {r.real_mean:.4f} (pseudo mean: {r.pseudo_mean:.4f})')
    print(f'\tstandardized difference: {r.cohens_d:.4f}')
    print(f'\tp-values: pre-FDR {r.welch_p:.6f} & post-FDR {r.welch_p_adj:.6f}')

		 VALIDATION (grand average, real vs pseudo)
			    using alpha of a=0.05


post-FDR-correction: 4/20 AUs w/ significant synchrony

AU06:
	mean: 0.2366 (pseudo mean: 0.2072)
	standardized difference: 0.7473
	p-values: pre-FDR 0.000000 & post-FDR 0.000001
AU12:
	mean: 0.2362 (pseudo mean: 0.2123)
	standardized difference: 0.6572
	p-values: pre-FDR 0.000001 & post-FDR 0.000008
AU25:
	mean: 0.2024 (pseudo mean: 0.1916)
	standardized difference: 0.3465
	p-values: pre-FDR 0.008717 & post-FDR 0.043585
AU26:
	mean: 0.1748 (pseudo mean: 0.1658)
	standardized difference: 0.3695
	p-values: pre-FDR 0.006407 & post-FDR 0.042716

2 AUs with significant pre-correction p-values.

AU02:
	mean: 0.1730 (pseudo mean: 0.1666)
	standardized difference: 0.2922
	p-values: pre-FDR 0.028724 & post-FDR 0.114896
AU24:
	mean: 0.1987 (pseudo mean: 0.1906)
	standardized difference: 0.2654
	p-values: pre-FDR 0.043973 & post-FDR 0.146577


		 VALIDATION (lag zero mean, real vs pseudo)
			    using alpha of a=0.05




In [10]:
# directional: does the signed synchrony lead correlate with enjoyment difference?

print('\n')
print('=' * 80)
print(f'{'\t' * 2} DIRECTIONAL (signed_lead_mean vs enjoy_diff)')
print(f'{'\t' * 3}    using alpha of a={ALPHA}')
print('=' * 80)
print('\n')

d = dir_res[dir_res['metric'] == 'signed_lead_mean']
sig = d[d['spearman_p_adj'] < ALPHA]
dropped = d[(d['spearman_p'] < ALPHA) & (d['spearman_p_adj'] >= ALPHA)]

print(f'\npost-FDR-correction: {len(sig)}/{len(d)} AUs significant\n')
if len(sig) > 0:
  for r in sig.itertuples():
    print(f'{r.au}:')
    print(f'\tspearman r: {r.spearman_r:.4f}')
    print(f'\tspearman p-values: pre-FDR {r.spearman_p:.4f} & post-FDR {r.spearman_p_adj:.4f}')
    print(f'\tsign agreement: {r.sign_agree:.6f}')

if len(dropped) > 0:
  print(f'\n{len(dropped)} AUs with significant pre-correction p-values.\n')
  for r in dropped.itertuples():
    print(f'{r.au}:')
    print(f'\tspearman r: {r.spearman_r:.4f}')
    print(f'\tspearman p-values: pre-FDR {r.spearman_p:.4f} & post-FDR {r.spearman_p_adj:.4f}')
    print(f'\tsign agreement: {r.sign_agree:.6f}')

print(f'\nsign agreement across all AUs (0.5 = chance): {d['sign_agree'].mean():.4f}')

print('\n')
print('=' * 80)
print(f'{'\t' * 2} DIRECTIONAL (best_lag_mean vs enjoy_diff)')
print(f'{'\t' * 3}    using alpha of a={ALPHA}')
print('=' * 80)
print('\n')

sig2 = dir_res[(dir_res['metric'] == 'best_lag_mean') & (dir_res['spearman_p_adj'] < ALPHA)]

print(f'{len(sig2)}/20 AUs significant')
if len(sig2) > 0:
  for r in sig2.itertuples():
    print(f'{r.au}:')
    print(f'\tspearman r: {r.spearman_r:.4f}')
    print(f'\tspearman p-values: pre-FDR {r.spearman_p:.4f} & post-FDR {r.spearman_p_adj:.4f}')
    print(f'\tsign agreement: {r.sign_agree:.6f}')



		 DIRECTIONAL (signed_lead_mean vs enjoy_diff)
			    using alpha of a=0.05



post-FDR-correction: 0/20 AUs significant


1 AUs with significant pre-correction p-values.

AU04:
	spearman r: -0.1888
	spearman p-values: pre-FDR 0.0373 & post-FDR 0.6072
	sign agreement: 0.393443

sign agreement across all AUs (0.5 = chance): 0.5025


		 DIRECTIONAL (best_lag_mean vs enjoy_diff)
			    using alpha of a=0.05


0/20 AUs significant


In [11]:
for metric in null_dir_res['metric'].unique():
    rows = null_dir_res['metric'] == metric
    null_dir_res.loc[rows, 'p_adj'] = stats.false_discovery_control(null_dir_res.loc[rows, 'p'])

In [12]:
# directional null check: pseudopairs should have essentially zero signed synchrony

print('\n')
print('=' * 80)
print(f'{'\t' * 2} DIRECTIONAL SYNCHRONY (pseudopair null check)')
print(f'{'\t' * 3}    using alpha of a={ALPHA}')
print('=' * 80)
print('\n')

nd = null_dir_res[null_dir_res['metric'] == 'signed_lead_mean']
sig = nd[nd['p_adj'] < ALPHA]
print(f'{len(sig)}/{len(nd)} AUs significantly ≠ 0\n')
if len(sig) > 0:
  for r in sig.itertuples():
    print(f'{r.au}:')
    print(f'\tmean: {r.mean:.4f}')
    print(f'\tp-value: {r.p:.6f}')
print(f'\nmean of pseudopairs\' signed lead across AUs: {nd['mean'].mean():.4f}')



		 DIRECTIONAL SYNCHRONY (pseudopair null check)
			    using alpha of a=0.05


1/20 AUs significantly ≠ 0

AU23:
	mean: 0.0087
	p-value: 0.000413

mean of pseudopairs' signed lead across AUs: 0.0015
